# Semantic Contamination in Telemetry: how a “mock” compromised clinical inferences at scale

*A practical study on inferential failures in LLMs and the need for sovereign architectures in digital health.*

---

**Renato Valente Ferreira**\
**T-Shaped HITL AI Systems Engineer**

---

The increasing adoption of large language models (LLMs) in digital health pipelines and IoT-based systems has introduced a new class of risks that are not yet fully characterized in applied literature: inferential failures induced by semantic contamination in data streams.

During an architectural audit of a ~60GB kinetic telemetry dataset from Parkinson’s patients — associated with the Michael J. Fox Foundation — a controlled experiment was conducted within an Edge AI pipeline (NSCP-L7) to evaluate LLM inferential behavior under partially synthetic inputs.

The intervention consisted of injecting a standardized “mock” string into the data payload — a common practice in development environments for pipeline validation.

The initial hypothesis was that models would prioritize real dataset signals, such as mechanical entropy metrics and frame duration, while ignoring non-correlated artificial elements.

### Data compliance and ethics note

> The dataset used in this experiment was fully anonymized, with no direct or indirect patient identifiers, in accordance with international best practices for privacy and research data usage.
> 
> Therefore, the observed phenomenon does not stem from sensitive data exposure, but rather from a structural limitation in how AI models interpret context when subjected to semantic interference.

### Targeted Prompt Contamination
Contrary to expectations, the models did not exhibit behavior consistent with evidence-driven clinical systems.

Instead, they operated as narrative coherence engines, attempting to justify the presence of the mock string at the expense of the underlying physiological signals.

As a result, more than 900 previously stable patients were associated with severe, non-existent diagnoses, including “Freezing Apneas,” without any support from the biomechanical data.

This behavior represents a phenomenon distinct from generic AI hallucination. We propose the term:

**Targeted Prompt Contamination in Telemetry**

The implications are substantial.

Routing raw sensor data directly to LLMs — particularly via cloud endpoints — without an intermediate layer of semantic and structural validation introduces a systemic risk vector.

### Adversarial validation as a robustness mechanism

Paradoxically, the same mechanism that introduces risk can be leveraged for validation.

The deliberate injection of semantically impossible scenarios enables the assessment of model grounding:

* Models that attempt to accommodate noise → **indicate inferential fragility**
* Models that detect inconsistency → **demonstrate contextual validation capability**

### The need for sovereign architectures

These findings point to the necessity of architectural reconfiguration.

Digital health systems cannot rely solely on centralized inference.

A sovereign control layer must be introduced between raw data and inference mechanisms, including:

* Edge-level filtering
* Semantic firewalls
* Explicit grounding mechanisms
* **Human-in-the-loop supervision**

This approach defines a distributed architecture with active governance at the edge, rather than exclusively in the cloud.

### Conclusion

The primary risk of AI in healthcare lies not only in model accuracy, but in how models construct coherence from imperfect inputs.

Language models are optimized for plausibility, not clinical truth.

Without additional validation mechanisms, errors may emerge with the appearance of consistency — making them particularly dangerous in critical contexts.

The future of AI in healthcare will depend less on standalone model sophistication and more on the robustness of the surrounding architectures.

In this context, human presence in the decision loop shifts from optional to structural.

---
### Appendix: Sovereign Data Sanitization (Code implementation)
To resolve the prompt contamination without passing data back through vulnerable cloud LLMs, the Sovereign HITL Engine executes offline regex filtering to strip the hallucination and retrieve the real clinical math (Mechanical Entropy / Telemetry Framing vs Freezing of Gait ratio).

In [1]:
import pandas as pd
import re
import os

# Note: On Kaggle, you would load this from your Dataset path (e.g. /kaggle/input/mjff-telemetry/Dossie_Clinico_MJ_Fox.csv)
input_file = 'Dossie_Clinico_MJ_Fox.csv' # Replace with Kaggle Path
output_file = 'Dossie_Sanitizado_MJ_Fox.csv'

# Regex patterns to extract the true L7 diagnostic data from the hallucinated text
regex_duracao = re.compile(r'DuracaoFramesRegistro[^0-9]+([0-9]{3,7})', re.IGNORECASE)
regex_entropia = re.compile(r'EntropiaMecanicaAcelerometro_Bloqueada_L7[^0-9]+([0-9]{3,7})', re.IGNORECASE)

records = []
if os.path.exists(input_file):
    df = pd.read_csv(input_file, sep='|', on_bad_lines='skip')
    
    for index, row in df.iterrows():
        try:
            paciente = row['ID_Paciente']
            laudo = str(row['Laudo_Clinico_Gemini_Flash']) + str(row['Laudo_Clinico_OpenAI'])
            
            # Extracting pure math from the LLM prose
            duracao_match = regex_duracao.search(laudo)
            entropia_match = regex_entropia.search(laudo)
            
            if duracao_match and entropia_match:
                duracao = int(duracao_match.group(1))
                entropia = int(entropia_match.group(1))
                
                # Calculate Severity (Freezing of Gait Ratio) avoiding LLM inference
                fog_ratio = (entropia / duracao) * 100 if duracao > 0 else 0
                    
                # Pure Mathematical Classification
                if fog_ratio > 90:
                    status = "Severe (Akinetic Crisis / Generalized Freezing)"
                elif fog_ratio > 50:
                    status = "Moderate (Frequent Blocking)"
                elif fog_ratio > 10:
                    status = "Mild (Intermittent Episodes)"
                else:
                    status = "Normal / Noise Filtered"
                    
                records.append({
                    'Patient_ID': paciente,
                    'Total_Frames': duracao,
                    'L7_Freezing_Alarm_Frames': entropia,
                    'Immobility_Ratio_Pct': round(fog_ratio, 2),
                    'Clinical_Status': status,
                })
        except Exception as e:
            pass

    sanitized_df = pd.DataFrame(records)
    sanitized_df.to_csv(output_file, index=False)
    print(f"Success! {len(sanitized_df)} patients scientifically recovered offline without LLM APIs.")
else:
    print("Please upload the corrupted dataset to the Kaggle environment to run the sanitization.")

Please upload the corrupted dataset to the Kaggle environment to run the sanitization.
